# 04: Gradient diagnostics from the saved E08 checkpoint

Use a fresh Colab GPU runtime. Run setup once, then work through the remaining cells.
Setup fetches the latest `main`; each diagnostic records the Git commit actually used.
The helper files must first be available on GitHub `main`.

This run repeats 30 forgetting updates from the original checkpoint using the
same settings as the previous 30-step run. It adds gradient direction,
actual Adam update sizes, and a before/after audit of model components.
An earlier JSON report cannot supply the new component measurements.


## 1. Reusable setup

Run once per fresh session. To update code after importing `uncle`, restart the runtime.


In [ ]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
_ = runpy.run_path("/content/uncle_colab_setup.py", run_name="__main__")


## 2. Experiment configuration

Change paths and experiment settings here.


In [ ]:
from pathlib import Path

CHECKPOINT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/before_forgetting.pt")
DATA = Path("/content/data/tiny-imagenet-200")
OUTPUT = CHECKPOINT.parent / "diagnostics" / "alignment_30_steps"
SETTINGS = dict(task="3", forgetting_lr=1e-5, gamma=5e-6, steps=30)
EXPECTED = dict(
    expected_history=[("learn", "3"), ("learn", "0")],
    expected_accuracies={"3": 26.0, "0": 44.6},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)


## 3. Reusable session and checkpoint checks

This mounts Drive and checks saved metadata. The diagnostic evaluates the actual
restored model at step zero. A metadata check is not a new accuracy measurement.


In [ ]:
from uncle.notebook import prepare_session

session = prepare_session(CHECKPOINT, **EXPECTED)


## 4. Run the diagnostic

Images are downloaded to runtime-local storage when needed. Every call restores
the original checkpoint and creates a separate report on Drive. Rerunning starts
again from that checkpoint. The original checkpoint is not overwritten.


In [ ]:
from uncle import diagnose_forgetting

report = diagnose_forgetting(
    CHECKPOINT, root=DATA, download=True, output=OUTPUT, device="cuda",
    audit_components=True, **SETTINGS,
)


## 5. Reusable results display

The accuracy table is measured after each update; loss gradients and raw-output
norms and cosine similarity are measured before it. Adam update norms are measured
after the optimizer step. A cosine near -1 means the terms oppose each other;
near +1 means they align. A zero initial preservation gradient has no cosine.
Cosine and update size still do not prove why accuracy changes.

The existing screen requires initial target and retained accuracy of at least
25%, target accuracy at most 12%, and retained drift below five percentage points.
A passing step is a candidate for further validation, not proof of deletion.
Share these tables and the report path before selecting another experiment.


In [ ]:
from uncle.notebook import show_diagnostic

screen = show_diagnostic(report, gradient_steps=(20, 25, 27, 30))

from uncle.research_diagnostic import assess_forgetting, runtime_run
assessment = assess_forgetting(runtime_run(report))
print("Best target accuracy within retention limit:",
      assessment["best_target_accuracy_within_retention_limit_pct"],
      "at step", assessment["best_step_within_retention_limit"])
print("Component changes (start to finish):", report["component_audit"])
